In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]  # adjust based on output
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

from ulta_sentiment import show_dashboard, show_overview 

In [3]:
from ulta_sentiment import _load_data, _data
import pandas as pd

_load_data()

brand_health = _data["brand_agg"].copy()
brand_health["health_score"] = (
    brand_health["avg_sentiment"] * 0.3
    + brand_health["rating_sentiment_alignment"] * 0.25
    + brand_health["sentiment_polarization"] * 0.25
    + (1 - brand_health["pct_1_star"]) * 0.2
)
brand_health = brand_health.sort_values("health_score", ascending=False).reset_index(drop=True)
brand_health.index += 1
brand_health.index.name = "Rank"

display_cols = ["brand", "health_score", "avg_sentiment", "avg_rating",
                "total_mismatch_rate", "n_reviews", 
                "complaint_concentration_label", "value_driver_label"]

pd.set_option("display.max_rows", None)
brand_health[display_cols].style.format({
    "health_score": "{:.3f}",
    "avg_sentiment": "{:.3f}",
    "avg_rating": "{:.2f}",
    "total_mismatch_rate": "{:.1%}",
    "n_reviews": "{:,.0f}",
}).background_gradient(subset=["health_score"], cmap="RdYlGn")

Loading Ulta dashboard data from CSVs...
  Loading reviews (subset columns)...
  Indexing by brand...
  Loaded 546 brands, 801,906 reviews in 1.6s


,brand,health_score,avg_sentiment,avg_rating,total_mismatch_rate,n_reviews,complaint_concentration_label,value_driver_label
Rank,,,,,,,,
1,beautyblender,0.929,0.792,4.89,1.7%,"1,231",concentrated,price_neutral
2,Kaja,0.927,0.781,4.83,1.9%,"1,458",concentrated,price_neutral
3,Kaja,0.927,0.781,4.83,1.9%,"1,458",concentrated,price_neutral
4,beautyblender,0.924,0.784,4.84,2.3%,"1,529",concentrated,price_neutral
5,Aramis,0.920,0.763,4.76,2.8%,218,insufficient_data,price_neutral
6,Aramis,0.920,0.763,4.76,2.8%,218,insufficient_data,price_neutral
7,NOYZ,0.919,0.755,4.66,3.1%,32,concentrated,price_neutral
8,Montblanc,0.917,0.758,4.84,2.4%,497,concentrated,price_neutral
9,OCTAVIA MORGAN LOS ANGELES,0.916,0.772,4.66,4.7%,601,concentrated,price_neutral


In [4]:
show_overview()

In [5]:
show_dashboard()

Dropdown(description='Brand:', layout=Layout(width='450px'), options=('18.21 Man Made', 'AG Care', 'ANUA', 'AR…

ToggleButtons(description='Segment:', options=(('All Reviews', 'all'), ('Verified Buyers', 'verified'), ('Unve…

Output()

## Ulta Brand Health Overview

### Methodology and Measurement Differences from Sephora

The Ulta brand health visualization uses a weighted combined sentiment metric — 70% body text plus 30% headline — rather than the pure VADER compound score used for Sephora. This design choice reflects a meaningful difference in review structure between the platforms: Ulta reviews include titled headlines that carry signal independent of the body text, and reviewers often use headlines as emotional summaries while reserving detail for the body. Weighting body text more heavily ensures that substantive product feedback dominates over the headline's often more effusive or headline-grabbing language, while still preserving the headline's contribution to overall sentiment characterization. The practical effect is that Ulta sentiment scores are compressed relative to Sephora's — the x-axis runs from approximately 0.30 to 0.80 combined sentiment, compared to Sephora's 0.40 to 1.0 VADER range — meaning direct cross-platform sentiment comparisons require adjusting for this methodological difference rather than treating the scales as equivalent.

The Ulta health ranking also covers a substantially larger catalog: the table runs to at least 1,416 rows before truncation, compared to Sephora's 304 brands, reflecting Ulta's much broader brand portfolio and the greater number of brands with sufficient review volume to generate meaningful health scores at the platform. This scale difference matters for interpreting the scatter: the Ulta bubble chart is representing a more heterogeneous set of brands, including mass-market drugstore brands, professional salon brands, mid-market accessible products, and prestige imports, all coexisting on the same axes. Sephora's chart by contrast represents a more pre-filtered prestige universe.

### Overall Shape and Main Cluster

The scatter follows the same positive diagonal as Sephora — higher sentiment brands receive higher ratings, and both correlate with lower mismatch rates in the deep-green color band. The main cluster of high-volume brands concentrates between 0.60 and 0.75 combined sentiment and between 4.4 and 4.8 stars, with the largest bubbles appearing in this zone. This operational core is solidly green, indicating that Ulta's most commercially significant brands have achieved the same coherence between written and numerical feedback that characterizes Sephora's best performers.

One notable structural feature of the Ulta scatter is that the main cluster sits lower on the sentiment axis than Sephora's equivalent cluster, even after accounting for the different sentiment methodology. The largest Ulta bubbles — representing brands with thousands of reviews like Valentino (2,994–3,077 reviews), NATASHA DENONA (5,472 reviews), Yves Saint Laurent (3,491 reviews), Sol de Janeiro (3,189–3,197 reviews), and Tanologist (3,157–3,247 reviews) — cluster in the 0.60–0.72 combined sentiment range. This reflects the broader and more democratized nature of Ulta's reviewer base: a platform that spans drugstore to prestige attracts reviewers whose written expression is more subdued on average than Sephora's more engaged specialist community. The sentiment compression is a platform culture effect rather than a product quality difference.

The large deep-green bubble visible near the top of the scatter at approximately 0.66 sentiment and 4.85 rating is I Dew Care, which ranks 12th on the health table with 2,777–2,796 reviews, a mismatch rate of 2.3%, and health scores of 0.915. Its prominence in the chart reflects a genuinely high-performing K-beauty brand that has built a large, coherent review base — consumers writing positively about it tend to also rate it highly, and the disconnect between the two is minimal.

### Top of the Health Ranking

The brands at the very top of Ulta's health table — beautyblender (0.929, rank 1), Kaja (0.927, ranks 2 and 3), Aramis (0.920), NOYZ (0.919), and Montblanc (0.917) — share a common characteristic with Sephora's top performers: they are either tool/accessory brands with objective performance criteria or fragrance brands with low efficacy-claim complexity. Beautyblender holds the top position with a mismatch rate of only 1.7% across 1,231 reviews — a sponge applicator whose primary evaluative criterion (does it blend better than alternatives) is easily and consistently settled by consumers, generating review-rating coherence. Kaja, an affordable K-beauty makeup brand, achieves the same coherence despite a very different product type, suggesting its accessible price point and whimsical product positioning set expectations that are reliably met rather than overshot or underdelivered. Aramis, a legacy men's fragrance brand, echoes the fragrance advantage seen across both platforms: concentrated complaint profiles and low mismatch.

The table's duplicate entries — beautyblender appearing at ranks 1, 4, and 25; Kaja at ranks 2 and 3; Aramis at ranks 5 and 6; and so on throughout — likely reflect the analysis being run across multiple product groupings or segmentation cuts within the same brand, with each segment receiving its own health score. The slight differences in review counts between duplicate entries (Kaja at 1,458 and 1,458, beautyblender at 1,231 and 1,529) confirm these are distinct subsets of the brand's review corpus rather than errors. This structural feature means the rank table should be read as brand-segment health rather than brand-level health, and aggregation across segments would produce a cleaner brand-level ranking.

### The Scatter's Lower-Left Tail

The brands below 0.50 combined sentiment and below 4.0 star rating form a sparse but diagnostically significant tail running toward the lower-left. These are predominantly small-volume brands — tiny bubbles rather than large ones — which makes sense structurally: Ulta's catalog depth means many brands accumulate small review counts on poor-performing or catalog-filler products before either gaining traction or disappearing from the assortment. The yellow and light-green coloring of many of these low-sentiment, low-rating outliers indicates moderate but not extreme mismatch rates, suggesting these are brands where consumers are consistently dissatisfied and express that dissatisfaction coherently in both their ratings and text rather than generating the specific incoherence pattern that high mismatch captures.

The two small red-orange dots visible at approximately 0.32–0.33 sentiment and 2.9–3.1 rating represent the lowest-health entries in the visible portion of the chart — brands where low sentiment, low rating, and elevated mismatch are all present simultaneously. These are the most structurally distressed positions in the Ulta catalog: products that consumers both rate poorly and write about inconsistently, meaning even the negative feedback is fragmented rather than pointing toward a specific addressable issue.

### Structural Comparison to Sephora

The most consequential difference between the two brand health scatters is the absence of red and orange coloration in Ulta's main cluster. In Sephora's chart, several large bubbles in the mid-tier of the main cluster — brands with meaningful review volume — showed orange and yellow-orange mismatch rates in the 0.15–0.25 range. In Ulta's chart, even the large mid-cluster bubbles are solidly green, with mismatch rates falling between 0 and 0.10. This is consistent with the DiD findings from Section 3.5, which showed Ulta's rating ATT (−0.263) is lower in magnitude than Sephora's (−0.328), and with the sentiment gap analysis showing Ulta generates fewer negative reviews proportionally. Ulta consumers appear to express dissatisfaction more directly and coherently than Sephora consumers — when they are unhappy, they say so in both their star rating and their written text, generating lower mismatch. Sephora's more engaged and critical reviewer culture produces more cases where consumers give diplomatically adequate star ratings while expressing sophisticated frustration in text, which is the mechanism that inflates Sephora's mismatch rates relative to Ulta's.

The practical implication for brands managing presence on both platforms is that mismatch-based intervention — identifying and resolving the gap between what consumers write and what they rate — is primarily a Sephora-channel problem. Ulta's lower mismatch rates mean that brands with elevated Ulta mismatch are genuine outliers worth investigating, while at Sephora a broader range of brands carry structurally elevated mismatch as a consequence of platform culture rather than brand-specific failure.